In [ ]:
#@title Install
#@markdown If you get a restart runtime warning, you don't have to, just click cancel
!git clone https://github.com/Kongchenglige/Minecraft_Skin_Generator.git
%cd Minecraft_Skin_Generator

# Install deps WITHOUT torch (preserve Colab's pre-installed CUDA version)
!grep -v '^torch$' Scripts/requirements_no_ui.txt > /tmp/req_no_torch.txt
!pip install -r /tmp/req_no_torch.txt
!apt-get install -y imagemagick
from IPython.display import clear_output
clear_output()
print("Installed!")

# Verify GPU + CUDA torch
import torch
print(f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # Sanity check: actually allocate on GPU
    t = torch.zeros(1, device='cuda')
    print(f"✅ CUDA tensor test passed (device={t.device})")
else:
    print("⚠️ No GPU found. Generation will be very slow.")

In [ ]:
#@title Upload Reference Image
from google.colab import files
import os

os.makedirs('input_images', exist_ok=True)

print("請上傳參考圖片（PNG/JPG/WebP）...")
uploaded = files.upload()

input_image_path = None
for filename in uploaded.keys():
    src = filename
    dst = f'input_images/{filename}'
    os.rename(src, dst)
    input_image_path = dst
    print(f"✅ Image saved: {dst}")

In [ ]:
#@title Generation Parameters

#@markdown Text prompt (combined with image guidance)
prompt = "a minecraft skin" #@param {type:"string"}

#@markdown IP-Adapter strength: higher = more like input image
ip_adapter_scale = 0.7 #@param {type:"slider", min:0.1, max:1.0, step:0.05}

#@markdown Inference steps (more = higher quality but slower)
num_inference_steps = 30 #@param {type:"integer"}

#@markdown Guidance scale
guidance_scale = 7.5 #@param {type:"number"}

#@markdown Precision type
model_precision_type = 'fp16' #@param ['fp16', 'fp32']

#@markdown Random seed (0 for random)
seed = 42 #@param {type:"integer"}

#@markdown Output filename
filename = "output-skin-from-image.png" #@param {type:"string"}

#@markdown View 3D model
model_3d = True #@param {type:"boolean"}

#@markdown Verbose output
verbose = False #@param {type:"boolean"}

In [ ]:
#@title Load Model (run once)
#@markdown 加載模型到 GPU，只需運行一次。之後可反復運行下方 Generate Cell。
import sys, logging
sys.path.insert(0, 'Scripts')

import torch
# Use importlib to handle hyphenated filename
import importlib
img2skin = importlib.import_module('minecraft-skins-sdxl-img2skin')

logging.basicConfig(stream=sys.stdout, level=logging.INFO, format='[%(asctime)s] %(levelname)s - %(message)s', force=True)
img2skin.logger = logging.getLogger('minecraft-skins-img2skin')
img2skin.logger.setLevel(logging.INFO)

dtype = torch.float16 if model_precision_type == 'fp16' else torch.float32
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

pipeline = img2skin.load_pipeline(device, dtype)
print('✅ Model loaded and ready!')

In [ ]:
#@title Generate Skin from Image (can re-run)
#@markdown 修改上方參數後可反復運行此 Cell，無需重新加載模型。
import os
from image_preprocessor import ImagePreprocessor

# 1. Preprocess
print(f'Preprocessing: {input_image_path}')
preprocessor = ImagePreprocessor()
reference_image = preprocessor.process(input_image_path)

# 2. Generate
print(f'Generating with prompt={prompt!r}, scale={ip_adapter_scale}')
generated_image = img2skin.generate_skin_from_image(
    pipeline=pipeline,
    reference_image=reference_image,
    prompt=prompt,
    ip_adapter_scale=ip_adapter_scale,
    num_inference_steps=num_inference_steps,
    guidance_scale=guidance_scale,
    seed=seed,
    device=device,
)

# 3. Extract skin
print('Extracting Minecraft skin...')
minecraft_skin = img2skin.extract_minecraft_skin(generated_image)

# 4. Save
os.makedirs('output_minecraft_skins', exist_ok=True)
output_path = os.path.join('output_minecraft_skins', filename)
minecraft_skin.save(output_path)
print(f'✅ Skin saved: {output_path}')

# 5. Optional 3D model
if model_3d:
    os.chdir('Scripts')
    ret = os.system(f"python to_3d_model.py '{filename}'")
    os.chdir('..')
    print('✅ 3D model generated' if ret == 0 else '⚠️ 3D model failed (non-critical)')

In [ ]:
#@title Display Generated Skin
import matplotlib.pyplot as plt

img = plt.imread(f"output_minecraft_skins/{filename}")
plt.figure(figsize=(8, 4))
plt.axis("off")
plt.title("Generated Minecraft Skin")
plt.imshow(img, interpolation='nearest')
plt.show()

In [ ]:
#@title Download Skin
from google.colab import files
files.download(f"output_minecraft_skins/{filename}")